## Practica Lab12: Aplicación del Flujo de Preprocesamiento para Machine Learning

## Dataset: Adult Income

**Download latest version**

path = kagglehub.dataset_download("uciml/adult-census-income")

---

## Objetivo

Aplicar de manera autónoma el flujo de preparación de datos para un problema de clasificación utilizando un conjunto de datos real.

Al finalizar la práctica, el estudiante será capaz de:

1. Importación de librerías
2. Carga del dataset
3. Comprensión del problema
4. Exploración inicial (EDA)
5. Correlación e hipótesis
6. Variables predictoras y objetivo
7. Tratamiento de nulos
8. One-Hot Encoding
9. Train/Test Split
10. StandardScaler
11. Árbol de Decisión
12. Predicciones
13. Accuracy
14. Feature Importance
15. Conclusiones

---

# Contexto del Problema

Una institución financiera desea analizar las características de distintos individuos para determinar qué factores están asociados con ingresos superiores a $50,000 dólares anuales.

El objetivo será construir un modelo capaz de predecir si una persona pertenece a uno de los siguientes grupos:


- (<=) 50K
- (>) 50K


## Preguntas

1. ¿Cuál es la variable objetivo?
2. ¿Qué representa dicha variable?
3. ¿Qué variables consideras que podrían influir más en el ingreso de una persona?
4. ¿Cuántas variables predictoras existen?
5. ¿Por qué fue necesario transformar variables categóricas?
7. ¿Cuántas columnas adicionales se generaron después del One-Hot Encoding?
8. ¿Existen valores nulos?
9. ¿Qué variables son numéricas?
10. ¿Qué variables son categóricas?
11. ¿Cuántos registros quedaron en entrenamiento?
12. ¿Cuántos registros quedaron en prueba?
13. ¿Por qué no debemos entrenar utilizando todos los datos?
14. ¿Cuál fue el Accuracy obtenido?
15. ¿Consideras que el resultado es adecuado?
16. ¿Qué factores podrían afectar el desempeño del modelo?
18. ¿Cuál fue la variable más importante?
19. ¿Cuál fue la menos importante?
20. ¿Coinciden los resultados con tus hipótesis iniciales?
21. ¿Qué variables aportan más información al modelo?


# Entregables

El repositorio deberá contener:

```text
Notebooks/
└── Laboratorio12.ipynb
```

---


## 1 Importación de librerías

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

import seaborn as sns
import matplotlib.pyplot as plt
import kagglehub

## 2 Carga del dataset

In [9]:
path = kagglehub.dataset_download("uciml/adult-census-income")
print("Ruta del dataset:", path)

Ruta del dataset: C:\Users\alexb\.cache\kagglehub\datasets\uciml\adult-census-income\versions\3


In [4]:
# Carga del DataFrame
dfAdult = pd.read_csv(path + "/adult.csv")
dfAdult.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


## 3 Comprensión del problema
**Contexto:** Una institución financiera desea analizar las características de distintos 
individuos para determinar qué factores están asociados con ingresos superiores a $50,000 dólares anuales.

**Objetivo:** Construir un modelo de clasificación binaria capaz de predecir si una persona 
pertenece al grupo de ingresos (<=50K) o (>50K).

## 4 Exploración inicial (EDA)

In [12]:
# Ver información general de tipos de datos y registros
dfAdult.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [13]:
# Ver la distribución inicial de la variable objetivo
print(dfAdult["income"].value_counts())

income
<=50K    24720
>50K      7841
Name: count, dtype: int64


In [14]:
# Estadísticas descriptivas de las variables numéricas
dfAdult.describe()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


## 5 Correlación e hipótesis
Hipótesis: A mayor nivel educativo (education-num), mayores serán los ingresos.

In [15]:
# Convertimos temporalmente income a numérico para ver correlación en el EDA
dfAdult['income_num'] = dfAdult['income'].apply(lambda x: 1 if ">50K" in str(x) else 0)

In [16]:
# Ver correlación de las variables numéricas con el ingreso
print(dfAdult.corr(numeric_only=True)['income_num'].sort_values(ascending=False))

income_num        1.000000
education.num     0.335154
age               0.234037
hours.per.week    0.229689
capital.gain      0.223329
capital.loss      0.150526
fnlwgt           -0.009463
Name: income_num, dtype: float64


In [17]:
# Borramos la columna temporal para no alterar 
dfAdult.drop(columns=['income_num'], inplace=True)

## 6. Variables predictoras y objetivo

In [34]:
# Variable objetivo (y) binaria
y = dfAdult["income"].apply(lambda x: 1 if ">50K" in str(x) else 0)

In [35]:
# Variables predictoras (X)
X = dfAdult.drop(columns=["income"])

## 7 Tratamiento de nulos

In [20]:
# Convertimos los caracteres '?' en nulos reales (NaN)
X.replace('?', np.nan, inplace=True)

In [36]:
# Rellenamos cada columna con su propia moda usando un bucle limpio
for col in X.columns:
    X[col].fillna(X[col].mode()[0], inplace=True)

C:\Users\alexb\AppData\Local\Temp\ipykernel_3192\942771370.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X[col].fillna(X[col].mode()[0], inplace=True)
C:\Users\alexb\AppData\Local\Temp\ipykernel_3192\942771370.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, w

In [37]:
# Confirmamos que el conteo de nulos sea 0 en todas las columnas
print(X.isnull().sum())

age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
dtype: int64


## 8 One-Hot Encoding

In [23]:
# Guardamos el número de columnas
columnas_antes = X.shape[1]

In [24]:
# Aplicamos One-Hot Encoding a todas las variables categóricas de texto
X = pd.get_dummies(X, drop_first=True)

print(f"Columnas originales: {columnas_antes} -> Columnas finales post-encoding: {X.shape[1]}")

Columnas originales: 14 -> Columnas finales post-encoding: 97


## 9 Train/Test Split

In [25]:
# División del dataset: 80% para entrenamiento y 20% para prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Registros en entrenamiento (X_train):", X_train.shape[0])
print("Registros en prueba (X_test):", X_test.shape[0])

Registros en entrenamiento (X_train): 26048
Registros en prueba (X_test): 6513


## 10 StandardScaler

In [26]:
# Instanciamos el escalador
scaler = StandardScaler()

In [27]:
# Ajustamos con el set de entrenamiento y transformamos ambos conjuntos
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 11 Árbol de Decisión

In [28]:
# Instanciamos el modelo limitando la profundidad para evitar sobreajuste extremo
clf = DecisionTreeClassifier(max_depth=10, random_state=42)

In [29]:
# Entrenamos el modelo con los datos escalados de entrenamiento
clf.fit(X_train_scaled, y_train)

DecisionTreeClassifier(max_depth=10, random_state=42)

## 12 Predicciones

In [30]:
# Realizamos las predicciones utilizando el conjunto de prueba escalado
y_pred = clf.predict(X_test_scaled)

## 13 Accuracy

In [31]:
# Calculamos y evaluamos el porcentaje de exactitud global del modelo
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy obtenido en el conjunto de prueba: {accuracy:.4f}")

Accuracy obtenido en el conjunto de prueba: 0.8537


## 14 Feature Importance

In [32]:
# Creamos un DataFrame con el peso informativo que el árbol le dio a cada variable
importancias = pd.DataFrame({
    'Variable': X.columns,
    'Importancia': clf.feature_importances_
})

In [38]:
# Mostramos las 10 variables que aportaron más información al modelo
importancias.sort_values(by="Importancia", ascending=False).head(10)

,Variable,Importancia
29,marital.status_Married-civ-spouse,0.382211
2,education.num,0.203737
3,capital.gain,0.182187
4,capital.loss,0.067354
0,age,0.051399
5,hours.per.week,0.033924
36,occupation_Exec-managerial,0.014669
1,fnlwgt,0.013586
10,workclass_Self-emp-not-inc,0.005599
42,occupation_Prof-specialty,0.005025


## 15 Conclusiones

### 1. Validación de Hipótesis Iniciales
Al contrastar las hipótesis planteadas en la exploración inicial con los resultados del modelo de Árbol de Decisión (`Feature Importance`), se determinaron las siguientes conclusiones:
* **Educación y Ganancias:** La hipótesis sobre la educación fue acertada; la variable `education-num` (años de estudio) se posicionó como uno de los factores con mayor peso informativo para el modelo. De igual forma, `capital-gain` demostró ser un indicador crítico directo de ingresos altos.
* El estado civil (`marital-status_Married-civ-spouse`) resultó ser la característica más determinante para el árbol. Esto nos indica que, estadísticamente en este conjunto de datos, la estabilidad del núcleo familiar o los hogares con doble ingreso juegan un papel crucial en la acumulación de un patrimonio superior a los $50,000 anuales.

### 2. Evaluación del Rendimiento del Modelo
* **Adecuación del Accuracy:** El modelo obtuvo un Accuracy de aproximadamente el **84-85%** en el conjunto de prueba. Consideramos que este resultado es **adecuado y sólido** para un modelo base (Baseline).
* **Factores que afectan el desempeño:** El rendimiento final se ve limitado principalmente por el **desbalanceo de clases** en la variable objetivo (`income`), dado que hay una proporción aproximada de 3 a 1 a favor de las personas que ganan `<=50K`. Esto tiende a sesgar al árbol hacia la clase mayoritaria.

### 3. Importancia del Flujo de Preprocesamiento
Este laboratorio demostró que el Machine Learning no depende solo del algoritmo, sino de la calidad de los datos previos:
* La correcta detección y sustitución de los caracteres ocultos `?` por la moda evitó introducir ruido en el entrenamiento.
* La expansión de variables mediante `One-Hot Encoding` nos permitió procesar variables cualitativas de gran valor (como ocupación o estado civil) que de otra forma habrían sido ignoradas por la librería `scikit-learn`.
* Finalmente, mantener un conjunto de prueba (`X_test`) aislado garantizó que la evaluación del Accuracy sea realista y libre de sobreajuste (overfitting).

# Preguntas

1. ¿Cuál es la variable objetivo?
La variable objetivo es income.

2. ¿Qué representa dicha variable?
Representa el nivel de ingresos anuales de un individuo, clasificado de forma binaria en dos categorías posibles: <=50K (ingresos menores o iguales a $50,000 dólares al año) o >50K (ingresos superiores a $50,000 dólares al año).

3. ¿Qué variables consideras que podrían influir más en el ingreso de una persona?
Las variables que más podrían influir son el nivel educativo (education o education-num), ya que a mayor preparación se accede a mejores sueldos; la edad (age), como un indicador de experiencia laboral; y las horas trabajadas por semana (hours-per-week).

4. ¿Cuántas variables predictoras existen?
En el dataset original existen 14 variables predictoras (las características o features que usamos para alimentar al modelo, excluyendo la columna income).

5. ¿Por qué fue necesario transformar variables categóricas?
Porque los algoritmos de Machine Learning de la librería scikit-learn (como DecisionTreeClassifier) son modelos matemáticos que solo entienden y procesan matrices puramente numéricas. No pueden interpretar directamente cadenas de texto como "Private", "Bachelors" o "Manager", por lo que es obligatorio mapearlas a números.

6. ¿Cuántas columnas adicionales se generaron después del One-Hot Encoding?
El dataset original tiene 14 predictoras. Tras aplicar pd.get_dummies(X, drop_first=True), las variables de texto se expanden horizontalmente (creando una columna binaria por cada categoría única). Dependiendo de si se eliminan los registros con ? o no, el número total de columnas finales en X sube a aproximadamente 90 - 100 columnas, lo que significa que se generaron alrededor de 75 a 85 columnas adicionales.

7. ¿Existen valores nulos?
Sí, existen. Sin embargo, vienen ocultos o camuflados en el dataset original bajo el carácter de signo de interrogación (?). Se encuentran concentrados principalmente en las variables categóricas workclass, occupation y native-country. Para manejarlos adecuadamente, se tuvieron que transformar primero a nulos reales (NaN) y luego imputarlos con la moda de sus respectivas columnas.

8. ¿Qué variables son numéricas?
Las variables numéricas (tanto continuas como discretas) son:

age (Edad)

fnlwgt (Ponderador demográfico)

education-num (Años de estudio logrados)

capital-gain (Ganancias de capital)

capital-loss (Pérdidas de capital)

hours-per-week (Horas laboradas a la semana)

9. ¿Qué variables son categóricas?
Las variables cualitativas o de texto son:

workclass (Sector de empleo)

education (Nivel escolar en formato de texto)

marital-status (Estado civil)

occupation (Puesto u oficio)

relationship (Estructura familiar / parentesco)

race (Raza)

gender / sex (Género)

native-country (País de procedencia)

10. ¿Cuántos registros quedaron en entrenamiento?
Utilizando una división estándar del 80% para entrenamiento (con el dataset completo de 32,561 registros), quedan 26,048 registros en el set de entrenamiento (X_train).

11. ¿Cuántos registros quedaron en prueba?
Utilizando el 20% restante para la validación, quedan 6,513 registros en el set de prueba (X_test).

12. ¿Por qué no debemos entrenar utilizando todos los datos?
Porque si usamos el 100% de los datos para entrenar, no tendríamos manera de evaluar cómo se comportará el modelo en producción con clientes reales y nuevos. El algoritmo podría simplemente memorizar los datos de entrenamiento (provocando sobreajuste u overfitting), dándonos una falsa impresión de precisión perfecta que fallará al procesar nueva información.

13. ¿Cuál fue el Accuracy obtenido?
Al entrenar el Árbol de Decisión controlando la profundidad (por ejemplo, con un max_depth=10), el Accuracy obtenido en el conjunto de prueba ronda habitualmente entre el 84% y 85% (0.8537).

14. ¿Consideras que el resultado es adecuado?
Sí, es bastante adecuado. Un nivel de precisión del ~85% indica que el modelo es capaz de clasificar correctamente a la gran mayoría de los individuos utilizando sus datos socioeconómicos. Es un desempeño sólido para un modelo base inicial.

15. ¿Qué factores podrían afectar el desempeño del modelo?
El desbalanceo de clases: En este dataset hay aproximadamente 3 personas que ganan <=50K por cada 1 persona que gana >50K. Esta desigualdad puede hacer que el árbol tienda a predecir la clase mayoritaria con más facilidad.

La profundidad del árbol: Si no se limita (max_depth), el árbol creará ramificaciones infinitas memorizando el ruido del dataset, lo que desplomará el Accuracy en el set de prueba.

Valores atípicos (outliers): En variables como capital-gain existen valores extremos que pueden sesgar ciertas decisiones en los nodos.

16. ¿Cuál fue la variable más importante?
Al analizar el feature_importances_, la variable que de forma consistente aparece en primer lugar como la más determinante es marital-status_Married-civ-spouse (Estado civil: Casado con cónyuge civil) o capital-gain (Ganancias de capital).

17. ¿Cuál fue la menos importante?
Las variables menos importantes (con un peso cercano o igual a 0.0) son las columnas individuales generadas por el One-Hot Encoding que corresponden a países de origen muy específicos y con baja frecuencia en los datos, por ejemplo: native-country_Honduras, native-country_Hungary o native-country_Holand-Netherlands.

18. ¿Coinciden los resultados con tus hipótesis iniciales?
Coinciden parcialmente. Las variables de educación (education-num) y la capacidad financiera (capital-gain) resultaron ser de las más informativas, tal como se predijo. Sin embargo, el estado civil (marital-status) resultó ser una variable con un peso mucho más alto del esperado de forma intuitiva, revelando un patrón estadístico muy fuerte en el dataset.

19. ¿Qué variables aportan más información al modelo?
Las variables que aportan mayor ganancia de información (reduciendo la entropía o impureza de Gini en los niveles más altos del árbol) son:

marital-status_Married-civ-spouse

capital-gain

education-num

age